# Implement a Metropolis-Hasting Sampler

Do you remember it? it is a MCMC sampler and it is quite simple in logic.


Here is the algorithm (or you can check it from [doi:10.1093/mnras/stt2190](https://academic.oup.com/mnras/article/437/4/3918/1011939)):

1. Choose $\boldsymbol{x}_0 \in \Omega_\boldsymbol{\theta}$
2. For each step ($\boldsymbol{x}_i$) propose $\boldsymbol{x}_\text{trial}$ drawn from a symmetric distribution $q(\boldsymbol{x}_\text{trial}|\boldsymbol{x}_i)$ (a gaussian??)
3. Compute $k \equiv \text{min}[1, p(\boldsymbol{x}_\text{trial})/p(\boldsymbol{x}_i)]$ the new value is given by
    * $\boldsymbol{x}_{i+1} = \boldsymbol{x}_i$ with probability $(1-k)$
    * $\boldsymbol{x}_{i+1} = \boldsymbol{x}_\text{trial}$ with probability $k$
4. Iterate from (2) *until convergence*

# Material that we need

First of all the libraries:

In [ ]:
import numpy
import random
from matplotlib import pyplot as plt

In [ ]:
rng = numpy.random.default_rng(seed=666)

## Bayes Theorem:

Let me refresh you on this: an MCMC sampler uses the likelihood to sample the parameter space in search for a representation of the posterior:

$$P(\boldsymbol{\theta}|\boldsymbol{D}) \propto P(\boldsymbol{\theta}) \cdot \mathcal{L}_\boldsymbol{D}(\boldsymbol{\theta})$$ 

i.e. we drop the evidence as the intent is to move **walkers** around the parameter space and have a measure to test whether a new position is more likely than another.

If you remember from the statistics course, we do not use the likelihood directly but its logarithm (because it is more stable and because reasons).

But before looking on that, let's generate a dataset:

In [ ]:
# Choose the "true" parameters.
m_true = -0.9594
b_true = 4.294
f_true = 0.534

In [ ]:
# Generate some synthetic data from the model.
N = 50
xobs = numpy.sort(10 * rng.normal(size=N))
yerr = 0.1 + .5 * rng.normal(size=N)
yobs = m_true * xobs + b_true
yobs += numpy.abs(f_true * yobs) * rng.standard_normal(size=N)
yobs += yerr * rng.standard_normal(size=N)
yerr = numpy.abs(yerr)

In [ ]:
fig, ax = plt.subplots(1,1)

_ = ax.errorbar(xobs, yobs, yerr=yerr, fmt=".k", capsize=0)
x0 = numpy.linspace(xobs.min(), xobs.max(), 500)
_ = ax.plot(x0, m_true * x0 + b_true, "k", alpha=0.3, lw=3)
_ = ax.set(xlabel='x', ylabel='y')

## LogLikelihood and Prior

I feel particularly kind today therefore you will not have to implement the log-likelihood and log-prior yourself, do not get used to it.

In [ ]:
def log_likelihood(theta, xx, yy, ee):
    m, b, log_f = theta
    model = m * xx + b
    sigma2 = ee**2 + model**2 * numpy.exp(2 * log_f)
    return -0.5 * numpy.sum((yy - model) ** 2 / sigma2 + numpy.log(sigma2))

In [ ]:
prior = [
    [-2.0,0.0],
    [2.0,6.0],
    [-4.0, 4.0]
]

In [ ]:
def log_prior(theta, prior):
    m, b, log_f = theta
    mlim, blim, flim = prior
    if mlim[0] < m < mlim[1] and blim[0] < b < blim[1] and flim[0] < log_f < flim[1]:
        return 0.0
    return -numpy.inf

In [ ]:
def log_probability(theta, xx, yy, ee, prior):
    lp = log_prior(theta, prior)
    if not numpy.isfinite(lp):
        return -numpy.inf
    return lp + log_likelihood(theta, xx, yy, ee)

## The actual assignment

I want you to implement the algorithm detailed at the beginning of this notebook. Desiderata:

* I would like to have a class which implements the sampler, stores the positions visited by the walkers and whether a position has been accepted or not.
* The class should implement the best practices we have seen in class (e.g. implement a `__str__` dunder method for printing)
* the class should be documented
* the first thing an class instance method should do is to check whether the attributes that have been passed by the user (you are the user in this case) have the correct properties (raise an `AttributeError` if not)

**Hints:**
* **test** **test** **test** **test** **test** **test** every step, you are in a jupyter notebook, open a new cell and test each functionality before inserting it into the class implementation
* I have provided at the end the code snippet to plot the marginalized posterior, nonetheless, I can already tell you that it should be gaussian (as the dataset is gaussian), thus the **mean** and **standard deviation** are good estimators of the posterior.
* you can try to implement it with **a single walker for starting**, doing it with multiple workers can be an extension

**Bonus:**
* If you feel particularly productive, also wrap everything into a module or a package.

**The algorithm again here**
[doi:10.1093/mnras/stt2190](https://academic.oup.com/mnras/article/437/4/3918/1011939):

1. Choose $\boldsymbol{x}_0 \in \Omega_\boldsymbol{\theta}$
2. For each step ($\boldsymbol{x}_i$) propose $\boldsymbol{x}_\text{trial}$ drawn from a symmetric distribution $q(\boldsymbol{x}_\text{trial}|\boldsymbol{x}_i)$ (a gaussian??)
3. Compute $k \equiv \text{min}[1, p(\boldsymbol{x}_\text{trial})/p(\boldsymbol{x}_i)]$ the new value is given by
    * $\boldsymbol{x}_{i+1} = \boldsymbol{x}_i$ with probability $(1-k)$
    * $\boldsymbol{x}_{i+1} = \boldsymbol{x}_\text{trial}$ with probability $k$
4. Iterate from (2) *until convergence*

In [ ]:
class MHsampler () :
    
    step = 0.05

    def __init__ ( self, nwalkers, logprob, ndim, *args, **kwargs ) :
        pass

    def run ( self, nsteps, pstart=None ) :
        pass

    def acceptance_fraction ( self ) :
        pass
    
    def get_flat_chain ( self, burnin = 0 ) :
        pass

I guess that then we will have to instantiate this object:

```python
sampler = MHsampler( nwalkers=1, logprob=log_probability, ndim=3, "what goes here?" )
```

and after having a working istance, we will need to run the sampling:

```python
sampler.run( 100000, pstart="what goes here?" )
```

## Test test test!!

* and when implementation testing is finished, the final test is to check whether or not it worked:
```python
outchain = sampler.get_flat_chain(burnin=1000)
import corner
fig = corner.corner(
    outchain, labels = ["m", "b", "log(f)"], truths = [m_true, b_true, numpy.log(f_true)]
)
```
![](posterior.png)